# Introduction: M5


In [0]:
%sh
cd ../
pip install . 
# --force-reinstall --no-deps 
pip install s3fs yfinance

In [0]:
import sys
sys.path.append('.')
from DB_MA_finetuning_qkcvlm import *

start_time = time.time()

In [0]:
_v = 1
v_qkcv = 3

### Dataset Creation

In [0]:

config_qkcv = Config_qkcv(_v)
config_qkcv.v_qkcv=v_qkcv

_col_forecast=config_qkcv.get_col_forecast()
print(_col_forecast)

horizon = 28

### reader
import pyarrow.parquet as pq
import pandas as pd
import s3fs,os
fs = s3fs.S3FileSystem()

_p_base = 's3://'

_p_base_folder = os.path.join(_p_base, 'm5-forecasting-accuracy')
file_db=f"Database_db_m5_{_col_forecast}_.csv"

_df_calendar = pd.read_csv(os.path.join(_p_base_folder, 'calendar.csv'))
_df_sell_prices = pd.read_csv(os.path.join(_p_base_folder, 'sell_prices.csv'))

### features not used in original model
_features_static = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
_features_dynamic_cat = ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI',]
_features_dynamic = _features_dynamic_cat + [ 'wday', 'month', 'year', 'sell_price']

_use_staging_train = True # skip data proceeding
_save_to_staging = True # update staging file


In [0]:
### load data
import glob

num_splits = 10

if not _use_staging_train:
    _df_complete_numeric = pd.read_csv(os.path.join(_p_base_folder, 'sales_train_evaluation.csv'))

    _df_complete_numeric=_df_complete_numeric.melt(id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], 
                   var_name='d', 
                   value_name='num_orders')

    print(f"day.min {_df_complete_numeric.d.min()}, day.max {_df_complete_numeric.d.max()}, num_orders.min {_df_complete_numeric.num_orders.min()}, num_orders.max() {_df_complete_numeric.num_orders.max()}")

    unique_id = 'id'

    _df_complete_numeric['open_flag'] = 1
    _df_complete_numeric['num_orders'] = _df_complete_numeric.groupby(unique_id)['num_orders'].ffill()
    _df_complete_numeric.isnull().sum()

    _df_complete_numeric['num_orders'].fillna(0.0001, inplace=True)
    _df_complete_numeric['open_flag'].fillna(0, inplace=True)

    ### merge with dynamic features
    _df_complete_numeric = _df_complete_numeric.merge(_df_calendar,on=['d']).merge(_df_sell_prices,on=['store_id', 'item_id', 'wm_yr_wk'])
    _df_complete_numeric['date'] = pd.to_datetime(_df_complete_numeric['date'])
    print(f"_df_complete_numeric date max {_df_complete_numeric['date'].max()}")
    _df_complete_numeric.rename(columns={unique_id: 'unique_id',
                                'num_orders': 'y',
                                'date': 'ds',
                                }, inplace=True)

    _df_complete_numeric.columns
    _df_complete_numeric.head(4)
    _df_complete_numeric.unique_id.count()
    _df_complete_numeric.unique_id.nunique()

    gc.collect()

    for _c in ['y',] + _features_dynamic + _features_static:
        _df_complete_numeric[_c].fillna(0.0001, inplace=True)

    for _c in _features_dynamic_cat + _features_static:
        _df_complete_numeric[_c] = _df_complete_numeric[_c].astype('category').cat.codes

    if _save_to_staging:
        print(f"saving files, y max {_df_complete_numeric['y'].max()}, min {_df_complete_numeric['y'].min()}, ds max {_df_complete_numeric.ds.max()}")
        
        ### save _df_complete_numeric
        # Split the dataframe into smaller dataframes
        df_splits = np.array_split(_df_complete_numeric, num_splits)

        # Save each split into a separate parquet file
        for i, df_split in enumerate(df_splits):
            df_split.to_parquet(os.path.join(_p_base_folder, f'_df_complete_numeric_part_{i}.parquet'))

else:
    # Read all parquet files in the specified directory
    parquet_files =[]
    for i in range(num_splits):
        parquet_files.append(os.path.join(_p_base_folder, f'_df_complete_numeric_part_{i}.parquet'))
    print(f'parquet_files {parquet_files}')
    # Concatenate all the parquet files into a single dataframe
    _df_complete_numeric = pd.concat([pd.read_parquet(file) for file in parquet_files])

_df_complete_numeric.describe()

### _df_static
_df_static_numeric = _df_complete_numeric[['unique_id'] + _features_static].drop_duplicates()


Y_train_df = _df_complete_numeric.loc[(_df_complete_numeric.ds >= '2011-01-29') & (_df_complete_numeric.ds < '2016-04-25'), _features_dynamic + ['unique_id', 'ds', 'y']]
Y_test_df = _df_complete_numeric.loc[(_df_complete_numeric.ds >= '2016-04-25') & (_df_complete_numeric.ds <= '2016-05-22'), _features_dynamic + ['unique_id', 'ds', 'y']]

### Model Creation

In [0]:
model, hparams, tfm_config = get_model(_features_static, 
                                       config_qkcv,
                                       horizon,
                                       load_weights=True)


In [0]:

predictions, y_future, model_tunned, finetuner, ca, attention_score = prediction_pipeline(tfm_config, horizon, config_qkcv, Y_train_df, Y_test_df, _df_static_numeric, model)



In [0]:
if not config_qkcv.train_only:
    df_merged, pred_vals_tunc = post_predictions(predictions, y_future)

    print(_v)
    print(_col_forecast)
    print(f'{_v}_M5_{_col_forecast}')

    # Example usage
    wpe_func(df_merged, eval_horizon = [horizon],forecast='forecast')

    # Example usage
    calculate_matrix(df_merged)
    mae = calculate_mae(df_merged)
    print(f"Mean Absolute Error (MAE): {mae}")

In [0]:

print(f"Execution time: {(time.time() - start_time)/60:.2f} mins")

In [0]:
folder_path = f"{_p_base}/results"

filename = f"M5_{_v}_{_col_forecast}"
if finetuner!=-1:
    pd.DataFrame(finetuner.training_matrix).to_csv(f"{folder_path}/matrix_{filename}.csv", index=False)

df_merged.to_csv(f"{folder_path}/df_merged_{filename}.csv", index=False)

ca.to_csv(f"{folder_path}/ca_{filename}.csv", index=False)
attention_score.to_csv(f"{folder_path}/attention_score_{filename}.csv", index=False)
